# 06 · Reproducción del champion desde el Model Registry

Notebook **limpio e independiente**: no importa nada de los notebooks anteriores ni reutiliza variables. Solo hace esto:
1. lee el **snapshot** de la OBT y verifica su hash SHA-256 contra el registrado en la corrida;
2. carga el modelo desde **`models:/centinela-modelo@champion`**;
3. recalcula en la partición de prueba (jul-dic 2020) la **AUC-PR** y la **pérdida neta** con el umbral guardado en la corrida;
4. las compara con **`==`** contra las métricas registradas en la corrida final del champion.

El servidor de MLflow del PSet 3 tiene que estar levantado (`docker compose up -d` en `pset_3/`).

In [1]:
import hashlib, json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from sklearn.metrics import average_precision_score

RAIZ = Path.cwd().parent
URI = next((l.split("=", 1)[1].strip() for l in (RAIZ / ".env").read_text().splitlines() if l.startswith("MLFLOW_TRACKING_URI=")),
           "http://127.0.0.1:5055") if (RAIZ / ".env").exists() else "http://127.0.0.1:5055"
mlflow.set_tracking_uri(URI)
cliente = MlflowClient()

version = cliente.get_model_version_by_alias("centinela-modelo", "champion")
corrida = cliente.get_run(version.run_id)
print(f"models:/centinela-modelo@champion -> versión {version.version} | run_id {version.run_id} | "
      f"corrida '{corrida.data.tags.get('mlflow.runName')}' (fase={corrida.data.tags.get('fase')})")

models:/centinela-modelo@champion -> versión 1 | run_id 70bfd857902b4cc188d52a94d934c442 | corrida 'final_champion_hist_gradient_boosting' (fase=final)


In [2]:
parquet = RAIZ / "data" / "obt_snapshot.parquet"
h = hashlib.sha256()
with open(parquet, "rb") as f:
    for bloque in iter(lambda: f.read(1 << 20), b""):
        h.update(bloque)
sha_registrado = corrida.data.params["datos_sha256"]
print("hash del snapshot:", h.hexdigest())
print("hash registrado  :", sha_registrado, "| coinciden:", h.hexdigest() == sha_registrado)
assert h.hexdigest() == sha_registrado, "El snapshot no es el mismo con el que se entrenó y evaluó el modelo"

df = pd.read_parquet(parquet)
prueba = df[df["particion_temporal"] == "prueba"].sort_values(["trans_ts", "trans_num"]).reset_index(drop=True)
y = prueba["is_fraud"].astype(int).to_numpy()
monto = prueba["amt"].astype(float).to_numpy()
print("prueba:", prueba.shape, f"| fraude {100*y.mean():.3f} %")

hash del snapshot: 9be410f73fb8c090e7647fc27559665c28431d29a0652548d16d8b932d9e9bce
hash registrado  : 9be410f73fb8c090e7647fc27559665c28431d29a0652548d16d8b932d9e9bce | coinciden: True


prueba: (525661, 40) | fraude 0.383 %


In [3]:
modelo = mlflow.sklearn.load_model("models:/centinela-modelo@champion")
# Columnas de entrada según la firma registrada del modelo (no se importan de src/).
# get_model_info de MLflow 2.17 no resuelve aliases: se usa la versión a la que apunta @champion.
columnas = mlflow.models.get_model_info(f"models:/centinela-modelo/{version.version}").signature.inputs.input_names()
score = modelo.predict_proba(prueba[columnas])[:, 1]
umbral = float(corrida.data.params["umbral_decision"])
alerta = score >= umbral
auc_pr = float(average_precision_score(y, score))
perdida = float(monto[(y == 1) & ~alerta].sum() + 5.0 * ((y == 0) & alerta).sum())

auc_reg = corrida.data.metrics["prueba_auc_pr_prueba"]
perdida_reg = corrida.data.metrics["prueba_perdida_neta"]
iguales = (auc_pr == auc_reg) and (perdida == perdida_reg)
print(pd.DataFrame({"registrada": [auc_reg, perdida_reg], "reproducida": [auc_pr, perdida],
                    "identicas_==": [auc_pr == auc_reg, perdida == perdida_reg]}, index=["auc_pr_prueba", "perdida_neta_prueba"]).to_string())
print(f"umbral de decisión registrado: {umbral!r} | columnas de entrada según la firma: {len(columnas)}")
print("REPRODUCIDO:", iguales)
assert iguales

                      registrada  reproducida  identicas_==
auc_pr_prueba            0.96547      0.96547          True
perdida_neta_prueba  14649.03000  14649.03000          True
umbral de decisión registrado: 0.23999331074785862 | columnas de entrada según la firma: 21
REPRODUCIDO: True
